# Visualize low confidence joints

* Load videos and keypoints
* For each video/keypoint pair:
  * Find only relevant frames
  * for each of those frames:
    * Find low confidence joints (<= 0.6)
    * Draw points on image
    * Save drawn frame

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import imageio

VIDEO_DIR = '../data/Utvalda filminspelningar för IRAF analys/Dec 2025 sit-stå och stå-sitt'
KEYPOINT_DIR = '../data/processed/keypoints_cropped/movenet'
OUT_DIR = '../data/processed/low_conf_videos'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

In [2]:
video_dict = {}

for video in Path(VIDEO_DIR).iterdir():
    if video.is_file():
        name = video.stem
        keypoint_path = Path(KEYPOINT_DIR) / f'{name}_movenet.csv'
        keypoints = pd.read_csv(keypoint_path)
        video_tuple = (video, keypoints)
        video_dict[name] = video_tuple

print(len(video_dict))

10


In [3]:
k = video_dict['DJI_20250425092743_0028_D'][1]
k[k['frame'] == 497].iloc[0]

frame                        497.000000
time                           9.940000
left_shoulder_x                0.528831
left_shoulder_y                0.526262
left_shoulder_confidence       0.868604
right_shoulder_x               0.493790
right_shoulder_y               0.519678
right_shoulder_confidence      0.871240
left_elbow_x                   0.486276
left_elbow_y                   0.585378
left_elbow_confidence          0.599897
right_elbow_x                  0.475691
right_elbow_y                  0.574786
right_elbow_confidence         0.643512
left_wrist_x                   0.491577
left_wrist_y                   0.532825
left_wrist_confidence          0.419674
right_wrist_x                  0.493891
right_wrist_y                  0.527743
right_wrist_confidence         0.483815
left_hip_x                     0.503408
left_hip_y                     0.660392
left_hip_confidence            0.886185
right_hip_x                    0.477625
right_hip_y                    0.651343


In [4]:
def draw_keypoints_on_image(image, keypoints, confidence_limit=0.6):
  height, width, _ = image.shape
  aspect_ratio = float(width) / height
  fig, ax = plt.subplots(figsize=(12*aspect_ratio,12))
  fig.tight_layout(pad=0)
  ax.margins(0)
  ax.set_yticklabels([])
  ax.set_xticklabels([])
  plt.axis('off')

  confidences = keypoints[keypoints.index.str.endswith('_confidence')]
  low_conf_joints = [joint.removesuffix('_confidence') for joint in confidences.index[np.nonzero(confidences <= confidence_limit)]]

  x = np.array([keypoints[f"{j}_x"] for j in low_conf_joints]) * width
  y = np.array([keypoints[f"{j}_y"] for j in low_conf_joints]) * height
  ax.imshow(image)
  ax.scatter(x, y, c="#00ff00")
  fig.canvas.draw()
  image_from_plot = np.frombuffer(fig.canvas.tostring_argb(), dtype=np.uint8)
  image_from_plot = image_from_plot.reshape(fig.canvas.get_width_height()[::-1] + (4,))
  image_from_plot = image_from_plot[:, :, 1:4]
  plt.close(fig)

  return image_from_plot


def draw_keypoints_on_video(video_path, keypoints, out_path, confidence_limit=0.6):
  reader = imageio.get_reader(video_path)
  fps = reader.get_meta_data()['fps']
  
  start_frame = int(keypoints['frame'].iloc[0])
  end_frame = int(keypoints['frame'].iloc[-1])
  
  with imageio.get_writer(out_path, fps=fps) as writer:
    for i, frame in enumerate(reader):
      if i < start_frame:
        continue
      if i >= end_frame:
        break

      row = keypoints[keypoints['frame'] == i].iloc[0]
      image = draw_keypoints_on_image(frame, row, confidence_limit=confidence_limit)
      writer.append_data(image)

In [ ]:
for video_name, video_tuple in video_dict.items():
  video_path = video_tuple[0]
  keypoints = video_tuple[1]
  out_path = Path(OUT_DIR) / f'{video_name}_low_conf.mp4'

  draw_keypoints_on_video(video_path, keypoints, out_path)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (2133, 1200) to (2144, 1200) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
IMAGEIO FFMPEG_WRITER WARNIN